In [1]:
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
features_path = Path("./1_1_features_and_labels.xlsx")
df = pd.read_excel(features_path)

In [3]:
df

,Код-идентификатор обследуемого,Возраст обследуемого,Пол обследуемого,Расположение на листе (без особенностей),Расположение на листе (смещение вправо),Расположение на листе (смещение влево),Расположение на листе (смещение вверх),Расположение на листе (смещение вниз),Расположение на листе (размещение в углу),"Расположение на листе (изображение обрезано - есть место для дорисовки, но не дорисовано)",...,Фон (заштрихованное облако),Фон (пейзаж),Фон (пейзаж преобладает над изображением человека),Фон (дождь),Фон (солнце),Фон (земля),Фон (другое),Площадь штриховки,Нажим,Шея отделена от тела линией
0,10000,2001,2002,1,2,3,4,5,6,7,...,249,250,251,252,1058,1059,1060,1062,1063,253
1,ЧУВ6лм1кл_6,6,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
2,ЧУВ6лм1кл_5,6,мужской,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
3,ЧУВ6лм1кл_4,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
4,ЧУВ5лмдс_51,5,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8578,ЧУВ5лждс (76),5,женский,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8579,ЧУВ5лждс (75),5,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8580,ЧУВ6лмдс,6,мужской,NaN,NaN,NaN,Смещение вверх/4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8581,1,1,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,Пейзаж/5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Галстук


In [4]:
features = df.iloc[0]
print(f"Все признаки имеют уникальные названия: {features.nunique(dropna=False) == features.shape[0]}")

Все признаки имеют уникальные названия: True


In [5]:
idents = df.iloc[1:]["Код-идентификатор обследуемого"]
vc = idents.value_counts(dropna=False)
dups = vc[vc > 1].to_dict()

print(f"Кол-во повторяющихся значений: {len(dups)}")
print(f"Кол-во строк для удаления: {sum(dups.values())}")
dups

Кол-во повторяющихся значений: 61
Кол-во строк для удаления: 130


{'ЧУВ5лждс': 10,
 'САМ_4_11лж5кл_25': 2,
 'САМ_3_14лм8кл_37': 2,
 'САМ7лж1кл_3': 2,
 'САМ7лж1кл_1': 2,
 'САМ12лм6кл_43': 2,
 'САМ2_13лж7кл_30': 2,
 'САМ2_12лм6кл_71': 2,
 'САМ_4_10лм4кл_05': 2,
 'САМ2_11лж5кл_53': 2,
 'САМ_3_10лм4кл_91': 2,
 'САМ2_5лждс_7': 2,
 'САМ2_5лждс_6': 2,
 'СВ7лждс_3': 2,
 'САМ2_5лждс_4': 2,
 'САМ11лм5кл_84': 2,
 'САМ_3_10лм4кл_90': 2,
 'САМ_4_11лжош_06': 2,
 'ЧУВ6лмдс': 2,
 'САМ11лж5кл_23': 2,
 'САМ_3_10лм4кл_1': 2,
 'САМ2_5лждс_54': 2,
 'САМ_3_10лм4кл_32': 2,
 'САМ12лм6кл_19': 2,
 'САМ11лм5кл_79': 2,
 'САМ_3_10лм4кл_2': 2,
 'САМ_3_10лм4кл_3': 2,
 'ЧУВ12лм6кл_81': 2,
 'САМ_3_13лж7кл_65': 2,
 'САМ_4_11лм6кл_03': 2,
 'МО_5лждс_14': 2,
 'САМ12лм6кл_73': 2,
 'САМ7лж1кл_55': 2,
 'САМ_5_9лж3кл_50': 2,
 'САМ4гмдс': 2,
 'САМ_4_9лж3кл_03': 2,
 'САМ7лм1кл_25': 2,
 'САМ6лмдс_59': 2,
 'САМ6лмдс_58': 2,
 'САМ_5_7лм1кл_41': 2,
 'САМ6лмдс_57': 2,
 'САМ_3_9лж3кл_10': 2,
 'САМ7лм1кл_54': 2,
 'САМ_5_11лж6кл_3': 2,
 'САМ_3_12лж7кл_3': 2,
 'САМ_5_10лж3кл_2': 2,
 1: 2,
 'САМ_5_10л

In [6]:
old_shape = df.shape
old_shape

(8583, 311)

In [7]:
col = "Код-идентификатор обследуемого"

df = df[~df[col].isin(dups.keys())]
df

,Код-идентификатор обследуемого,Возраст обследуемого,Пол обследуемого,Расположение на листе (без особенностей),Расположение на листе (смещение вправо),Расположение на листе (смещение влево),Расположение на листе (смещение вверх),Расположение на листе (смещение вниз),Расположение на листе (размещение в углу),"Расположение на листе (изображение обрезано - есть место для дорисовки, но не дорисовано)",...,Фон (заштрихованное облако),Фон (пейзаж),Фон (пейзаж преобладает над изображением человека),Фон (дождь),Фон (солнце),Фон (земля),Фон (другое),Площадь штриховки,Нажим,Шея отделена от тела линией
0,10000,2001,2002,1,2,3,4,5,6,7,...,249,250,251,252,1058,1059,1060,1062,1063,253
1,ЧУВ6лм1кл_6,6,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
2,ЧУВ6лм1кл_5,6,мужской,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
3,ЧУВ6лм1кл_4,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
4,ЧУВ5лмдс_51,5,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8575,ЧУВ5лждс (79),5,женский,NaN,Смещение вправо/2,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,Пейзаж/5,NaN,NaN,Солнце,Земля,NaN,NaN,NaN,ничего из перечисленного
8576,ЧУВ5лждс (78),5,женский,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8577,ЧУВ5лждс (77),5,женский,NaN,Смещение вправо/2,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8578,ЧУВ5лждс (76),5,женский,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия


In [8]:
new_shape = df.shape
new_shape

(8453, 311)

In [9]:
old_shape[0] - sum(dups.values()), new_shape[1]

(8453, 311)

In [10]:
df.shape

(8453, 311)

In [11]:
unique_feature_values = {}
for col in df.columns[1:]:
    vals = pd.unique(df.iloc[1:][col])
    d = {}
    for v in vals:
        if pd.isna(v):
            d[None] = None
        else:
            s = f"{v}"
            d[s] = s
    unique_feature_values[col] = d
unique_feature_values

{'Возраст обследуемого': {'6': '6',
  '5': '5',
  '12': '12',
  '11': '11',
  '7': '7',
  '10': '10',
  '9': '9',
  '13': '13',
  '8': '8',
  '14': '14',
  '4': '4',
  '15': '15',
  '16': '16',
  '63': '63',
  '72': '72'},
 'Пол обследуемого': {'мужской': 'мужской', 'женский': 'женский'},
 'Расположение на листе (без особенностей)': {None: None,
  'Без особенностей/1': 'Без особенностей/1'},
 'Расположение на листе (смещение вправо)': {None: None,
  'Смещение вправо/2': 'Смещение вправо/2'},
 'Расположение на листе (смещение влево)': {'Смещение влево/3': 'Смещение влево/3',
  None: None},
 'Расположение на листе (смещение вверх)': {None: None,
  'Смещение вверх/4': 'Смещение вверх/4'},
 'Расположение на листе (смещение вниз)': {None: None,
  'Смещение вниз/5': 'Смещение вниз/5'},
 'Расположение на листе (размещение в углу)': {None: None,
  'Размещение в углу/6': 'Размещение в углу/6'},
 'Расположение на листе (изображение обрезано - есть место для дорисовки, но не дорисовано)': {None: 

In [12]:
def clean_unique_feature_value(val):
    if pd.isna(val):
        return None
    s = str(val).replace("\xa0", " ")
    if "=>" in s:
        s = s.split("=>", 1)[0]
    while True:
        s2 = s.strip()
        s2 = s2.rstrip(",").strip()
        s2 = re.sub(r"/\s*\d+\s*$", "", s2).strip()
        if s2 == s.strip():
            s = s2
            break
        s = s2
    s = s.replace("«", "'").replace("»", "'")
    s = s.replace('"', "'")
    s = s.strip().rstrip(",").strip()
    s = s.lower()
    s = re.sub(r"\s*/\s*", " / ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def unique_feature_values_jsonable(obj):
    if isinstance(obj, dict):
        return {
            ("null" if k is None else k): unique_feature_values_jsonable(v)
            for k, v in obj.items()
        }
    return obj


def sort_inner_feature_dict(d):
    cleaned = {k: clean_unique_feature_value(k) for k in d}
    out = {}
    for k in sorted((x for x in cleaned if x is not None), key=str):
        out[k] = cleaned[k]
    if None in cleaned:
        out[None] = cleaned[None]
    return out


unique_feature_values = {
    col: sort_inner_feature_dict(d) for col, d in unique_feature_values.items()
}

with open("unique_feature_values.json", "w", encoding="utf-8") as f:
    json.dump(
        unique_feature_values_jsonable(unique_feature_values),
        f,
        ensure_ascii=False,
        indent=2,
    )

In [13]:
cols = df.columns[1:]
idx = df.index[1:]
body = df.loc[idx, cols]
body

,Возраст обследуемого,Пол обследуемого,Расположение на листе (без особенностей),Расположение на листе (смещение вправо),Расположение на листе (смещение влево),Расположение на листе (смещение вверх),Расположение на листе (смещение вниз),Расположение на листе (размещение в углу),"Расположение на листе (изображение обрезано - есть место для дорисовки, но не дорисовано)",Расположение на листе (изоброжение обрезано - обрезан краем листа / нет места для дорисовки),...,Фон (заштрихованное облако),Фон (пейзаж),Фон (пейзаж преобладает над изображением человека),Фон (дождь),Фон (солнце),Фон (земля),Фон (другое),Площадь штриховки,Нажим,Шея отделена от тела линией
1,6,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
2,6,мужской,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
3,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Изоброжение обрезано - обрезан краем листа/ не...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
4,5,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
5,5,мужской,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8575,5,женский,NaN,Смещение вправо/2,NaN,NaN,Смещение вниз/5,NaN,NaN,NaN,...,NaN,Пейзаж/5,NaN,NaN,Солнце,Земля,NaN,NaN,NaN,ничего из перечисленного
8576,5,женский,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8577,5,женский,NaN,Смещение вправо/2,NaN,NaN,Смещение вниз/5,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8578,5,женский,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия


In [14]:
# Вносим дополнительные правки в признаки

#   "Размер глаз": {
#     "Большые/3": "большые",
#     "Маленькие/1": "маленькие",
#     "Среднийе (стандарт)/2": "среднийе (стандарт)",
#     "null": null
#   },

#   "Размер глаз": {
#     "Большые/3": "большые",
#     "Маленькие/1": "маленькие",
#     "Среднийе (стандарт)/2": "среднийе (стандарт)",
#     "null": null
#   },

#   "Тело (отсутствует)": {
#     "Отсутствует 7": "отсутствует 7",
#     "null": null
#   },

print(unique_feature_values["Размер глаз"])
unique_feature_values["Размер глаз"]["Большые/3"] = "большие"
unique_feature_values["Размер глаз"]["Среднийе (стандарт)/2"] = "средние (стандарт)"
print(unique_feature_values["Размер глаз"])

print(unique_feature_values["Тело (отсутствует)"])
unique_feature_values["Тело (отсутствует)"]["Отсутствует 7"] = "отсутствует"
print(unique_feature_values["Тело (отсутствует)"])

# with open("unique_feature_values.json", "w", encoding="utf-8") as f:
#     json.dump(
#         unique_feature_values_jsonable(unique_feature_values),
#         f,
#         ensure_ascii=False,
#         indent=2,
#     )

{'Большые/3': 'большые', 'Маленькие/1': 'маленькие', 'Среднийе (стандарт)/2': 'среднийе (стандарт)', None: None}
{'Большые/3': 'большие', 'Маленькие/1': 'маленькие', 'Среднийе (стандарт)/2': 'средние (стандарт)', None: None}
{'Отсутствует 7': 'отсутствует 7', None: None}
{'Отсутствует 7': 'отсутствует', None: None}


In [15]:
df = df.copy()
cols_map = [c for c in cols if c in unique_feature_values]
if cols_map:
    df[cols_map] = df[cols_map].astype(object)
body = df.loc[idx, cols]
for col in cols:
    if col not in unique_feature_values:
        continue
    m = unique_feature_values[col]
    s = body[col]

    def map_cell(v):
        if pd.isna(v):
            return v
        k = v if isinstance(v, str) else str(v)
        if k not in m:
            return v
        return m[k]

    df.loc[idx, col] = s.map(map_cell)

In [16]:
df

,Код-идентификатор обследуемого,Возраст обследуемого,Пол обследуемого,Расположение на листе (без особенностей),Расположение на листе (смещение вправо),Расположение на листе (смещение влево),Расположение на листе (смещение вверх),Расположение на листе (смещение вниз),Расположение на листе (размещение в углу),"Расположение на листе (изображение обрезано - есть место для дорисовки, но не дорисовано)",...,Фон (заштрихованное облако),Фон (пейзаж),Фон (пейзаж преобладает над изображением человека),Фон (дождь),Фон (солнце),Фон (земля),Фон (другое),Площадь штриховки,Нажим,Шея отделена от тела линией
0,10000,2001,2002,1,2,3,4,5,6,7,...,249,250,251,252,1058,1059,1060,1062,1063,253
1,ЧУВ6лм1кл_6,6,мужской,NaN,NaN,смещение влево,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,шея отсутствует
2,ЧУВ6лм1кл_5,6,мужской,NaN,NaN,NaN,NaN,смещение вниз,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,шея отсутствует
3,ЧУВ6лм1кл_4,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,шея отсутствует
4,ЧУВ5лмдс_51,5,мужской,NaN,NaN,смещение влево,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8575,ЧУВ5лждс (79),5,женский,NaN,смещение вправо,NaN,NaN,смещение вниз,NaN,NaN,...,NaN,пейзаж,NaN,NaN,солнце,земля,NaN,NaN,NaN,ничего из перечисленного
8576,ЧУВ5лждс (78),5,женский,NaN,NaN,смещение влево,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,просто линия
8577,ЧУВ5лждс (77),5,женский,NaN,смещение вправо,NaN,NaN,смещение вниз,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8578,ЧУВ5лждс (76),5,женский,без особенностей,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,просто линия


In [17]:
# # Теперь нужно понять, какие группы признаков мы можем объединить

# Расположение на листе (без особенностей)	
# Расположение на листе (смещение вправо)	
# Расположение на листе (смещение влево)	
# Расположение на листе (смещение вверх)	
# Расположение на листе (смещение вниз)	
# Расположение на листе (размещение в углу)	
# Расположение на листе (изображение обрезано - есть место для дорисовки, но не дорисовано)	
# Расположение на листе (изоброжение обрезано - обрезан краем листа / нет места для дорисовки)	

# Степень смещения вправо	
# Степень смещения влево	
# Степень смещения вверх	
# Степень смещения вниз	

# Размещение в углу	

# Что обрезано (голова)	
# Что обрезано (одна рука)	
# Что обрезано (обе руки)	
# Что обрезано (ноги)	

# Размер рисунка	

# Ракурс	

# Поза	

# Фигура в движении	

# Способ изображения	

# Изображение персонажа	

# Персонаж	

# Нажим (без особенностей)	
# Нажим (слабый)	
# Нажим (сильный)	
# Нажим (сильно варьирует)	

# Особенности линии (без особенностей)	
# Особенности линии (множественные)	
# Особенности линии (эскизные)	
# собенности линии ('кусочные')	
# Особенности линии (недоведенные)	
# Особенности линии ('промахивающиеся')	
# Особенности линии (другое)	

# Множественные, степень выраженности	
# Эскизные, степень выраженности	
# 'Кусочные', степень выраженности	
# Недоведенные, степень выраженности	
# 'Промахивающиеся', степень выраженности	

# Аккуратность рисунка	

# Штриховка	

# Нажим штриховки (без особенностей)	
# Нажим штриховки (слабый)	
# Нажим штриховки (сильный)	
# Нажим штриховки (варьирует)	

# Аккуратность штриховки (очень низкая)	
# Аккуратность штриховки (низкая)	
# Аккуратность штриховки (средняя (стандарт))	
# Аккуратность штриховки (высокая)	
# Аккуратность штриховки (очень высокая)	
# Аккуратность штриховки (разные виды штриховок)	

# Детализированность	

# Стирания, исправления	

# Устойчивость изображенного человека	

# Грубые нарушения рисунка (отсутствуют)	
# Грубые нарушения рисунка (отклонение от вертикали)	
# Грубые нарушения рисунка (нарушение пропорций)	
# Грубые нарушения рисунка (грубая асимметрия)	

# Изображение в рамке	Голова (без особенностей)	

# Голова (маленькая)	Голова (большая)	
# Голова (искажена форма)	
# Голова (зачерненная / заштрихованная)	
# Голова (отсутствует)	

# Детали (лицо) (глаза (один или оба))	
# Детали (лицо) (нос)	
# Детали (лицо) (рот)	
# Детали (лицо) (уши)	
# Детали (лицо) (лоб)	
# Детали (лицо) (волосы (шапка))	
# Детали (лицо) (другое)	
# Детали (лицо) (отсутствуют все указанные выше пункты)	

# Два глаза одинаковые	

# Форма глаз (точка)	
# Форма глаз (черточка)	
# Форма глаз (круг)	
# Форма глаз (овал, реалистичная форма)	
# Форма глаз («мультяшные»)	
# Форма глаз (форма искажена)	
# Форма глаз (отсутствуют)	

# Форма левого глаза (точка)	
# Форма левого глаза (черточка)	

# Форма левого глаза (круг)	
# Форма левого глаза (овал, реалистичная форма)	
# Форма левого глаза («мультяшные»)	
# Форма левого глаза (форма искажена)	
# Форма левого глаза (отсутствует)	

# Форма правого глаза (точка)	
# Форма правого глаза (черточка)	
# Форма правого глаза (круг)	
# Форма правого глаза (овал, реалистичная форма)	
# Форма правого глаза («мультяшные»)	
# Форма правого глаза (форма искажена)	
# Форма правого глаза (отсутствует)	

# Размер глаз	

# Размер левого глаза	

# Размер правого глаза	

# Дополнительные особенности глаз (без особенностей)	
# Дополнительные особенности глаз (подчеркнутый контур)	
# Дополнительные особенности глаз («пустые»)	
# Дополнительные особенности глаз (зачернена радужка или весь глаз)	
# Дополнительные особенности глаз (один или оба глаза закрыты волосами или шапкой)	
# Дополнительные особенности глаз (изображены очки (прозрачные))	
# Дополнительные особенности глаз (изображены темные очки)	
# Дополнительные особенности глаз (закрыты)	
# Дополнительные особенности глаз (полузакрыты (прищур))	

# Дополнительные особенности правого глаза (без особенностей)	
# Дополнительные особенности правого глаза (подчеркнутый контур)	
# Дополнительные особенности правого глаза («пустые»)	
# Дополнительные особенности правого глаза (зачернена радужка или весь глаз)	
# Дополнительные особенности правого глаза (закрыт волосами или шапкой)	
# Дополнительные особенности правого глаза (изображены очки (прозрачные))	
# Дополнительные особенности правого глаза (изображены темные очки)	
# Дополнительные особенности правого глаза (закрыт)	
# Дополнительные особенности правого глаза (полузакрыт (прищур))	

# Дополнительные особенности левого глаза (без особенностей)	
# Дополнительные особенности левого глаза (подчеркнутый контур)	
# Дополнительные особенности левого глаза («пустые»)	
# Дополнительные особенности левого глаза (зачернена радужка или весь глаз)	
# Дополнительные особенности левого глаза (закрыт волосами или шапкой)	
# Дополнительные особенности левого глаза (изображены очки (прозрачные))	
# Дополнительные особенности левого глаза (изображены темные очки)	
# Дополнительные особенности левого глаза (закрыт)	
# Дополнительные особенности левого глаза (полузакрыт (прищур))	

# Ресницы	Нос (без особенностей)	

# Нос (увеличен размер)	
# Нос (усилен нажим)	
# Нос (другое)	
# Нос (отсутствует)	

# Рот	

# Форма рта	

# Положение рта / губ	

# Зубы	

# Язык	

# Уши (без особенностей)	
# Уши (усилен нажим)	
# Уши (преувеличен размер)	
# Уши (другое)	
# Уши (отсутствуют)	

# Лоб (маленький)	
# Лоб (средний (стандарт))	
# Лоб (большой)	
# Лоб (скрыт волосами или шапкой)	
# Лоб (скрыт волосами, усилен нажим)	
# Лоб (другое)	
# Лоб (отсутствует)	

# Волосы или шапка (без особенностей)	
# Волосы или шапка (нарисованы небрежно)	
# Волосы или шапка (нарисованы особо старательно)	
# Волосы или шапка (другое)	
# Волосы или шапка (отсутствуют)	

# Дополнительные детали на лице (брови)	
# Дополнительные детали на лице (усы и / или борода)	
# Дополнительные детали на лице (морщины)	
# Дополнительные детали на лице (маска)	
# Дополнительные детали на лице (рисунки на лице)	
# Дополнительные детали на лице (другое)	
# Дополнительные детали на лице (отсутствуют)	

# Руки (одна или обе)	

# Положение рук (слегка отставлены (стандарт))	
# Положение рук (прижаты к телу (от верха до локтя))	
# Положение рук (свободно опущены)	
# Положение рук (сильно отставлены)	
# Положение рук (вытянуты в стороны или вперед)	
# Положение рук (подняты)	Положение рук (дополнительно)	
# Положение рук (за спиной или в карманах)	
# Положение рук (на фоне тела)	
# Положение рук (скрещены на груди)	
# Положение рук (уперты в бока)	
# Положение рук (закрывают генитальную область)	
# Положение рук (прижаты к голове)	
# Положение рук (другое)	

# Положение правой руки	

# Положение правой руки.1	

# Положение левой руки	

# Положение левой руки.1	

# Положение руки (слегка отставлена (стандарт))	
# Положение руки (прижата к телу (от верха до локтя))	
# Положение руки (свободно опущена)	
# Положение руки (сильно отставлена)	
# Положение руки (вытянута в сторону или вперед)	
# Положение руки (поднята)	
# Положение руки (дополнительно)	
# Положение руки (за спиной или в кармане)	
# Положение руки (на фоне тела)	
# Положение руки (уперта в бок)	
# Положение руки (закрывает генитальную область)	
# Положение руки (прижата к голове)	
# Положение руки (другое)	Кисть (маленькая)	

# Кисть (средняя (стандарт))	
# Кисть (большая)	
# Кисть (повышен нажим)	
# Кисть (зачернена)	
# Кисть (отсутствует)	

# Ладонь, пальцы (изображены пальцы)	
# Ладонь, пальцы (изображена ладонь)	
# Ладонь, пальцы (положение «варежки»)	
# Ладонь, пальцы (кулак, большой палец снаружи)	
# Ладонь, пальцы (кулак, большой палец не виден)	
# Ладонь, пальцы (отсутствуют)	

# Пальцы (изображены черточками)	
# Пальцы (передана толщина)	
# Пальцы (округлые ногти)	
# Пальцы (заостренные ногти)	
# Пальцы (другое)	

# Ноги (одна или обе) (две ноги в одинаковом положении)	
# Ноги (одна или обе) (две ноги в разном положении)	
# Ноги (одна или обе) (только одна нога)	
# Ноги (одна или обе) (скрыты препятствием)	
# Ноги (одна или обе) (отсутствуют)	
# Ноги (сведены вместе)	
# Ноги (слегка расставлены (стандарт))	
# Ноги (широко расставлены)	
# Ноги (короткие)	
# Ноги (длинные)	
# Ноги (другое)	

# Правая нога (вертикальна)	
# Правая нога (отставлена (стандарт))	
# Правая нога (согнута)	
# Правая нога (короткая)	
# Правая нога (длинная)	
# Правая нога (другое)	

# Левая нога (вертикальна)	
# Левая нога (отставлена (стандарт))	
# Левая нога (согнута)	
# Левая нога (короткая)	
# Левая нога (длинная)	
# Левая нога (другое)	

# Нога (вертикальна)	
# Нога (отставлена (стандарт))	
# Нога (согнута)	
# Нога (короткая)	
# Нога (длинная)	
# Нога (другое)	

# Ступни (без особенностей)	
# Ступни (маленькие)	
# Ступни (большие)	
# Ступни (повышен нажим)	
# Ступни (зачернены)	
# Ступни (отсутствуют)	
# Ступни (другое)	

# Шея (без особенностей)	
# Шея (одинарная линия)	
# Шея (тонкая)	
# Шея (толстая)	
# Шея (короткая)	
# Шея (длинная)	
# Шея (отсутствует)	

# Тело (палочка)	
# Тело (округлое)	
# Тело (прямоугольное)	
# Тело (реалистичное)	
# Тело (вытянутое)	
# Тело (широкое)	
# Тело (отсутствует)	
# Тело (треугольное)	
# Тело (другое)	

# Пол (женский)	
# Пол (мужской)	
# Пол (неопределенный)	
# Пол (признаки пола подчеркнуты)	
# Пол (подчеркнута генитальная область)	
# Пол (изображены половые органы)	

# Одежда (реалистичное изображение)	
# Одежда (условное изображение)	
# Одежда («рентгеновский» рисунок)	
# Одежда (отсутствует, тело изображено условно)	
# Одежда (отсутствует, человек обнажен)	

# Дополнительные детали изображения фигуры (отсутствуют)	
# Дополнительные детали изображения фигуры (пуговицы)	
# Дополнительные детали изображения фигуры (пуп)	
# Дополнительные детали изображения фигуры (ширинка)	
# Дополнительные детали изображения фигуры (волосы на теле или конечностях)	
# Дополнительные детали изображения фигуры (украшения)	
# Дополнительные детали изображения фигуры (другое)	
# Дополнительные детали изображения фигуры (ремень / пояс)	

# Пуговицы	

# Украшения, узор на одежде	

# Дополнительно (в руке или рядом) (игрушка, шарик)	
# Дополнительно (в руке или рядом) (цветок)	
# Дополнительно (в руке или рядом) (животное)	
# Дополнительно (в руке или рядом) (палка)	
# Дополнительно (в руке или рядом) (меч, нож, топор)	
# Дополнительно (в руке или рядом) (пистолет, ружье)	
# Дополнительно (в руке или рядом) (шприц)	
# Дополнительно (в руке или рядом) (сигарета)	
# Дополнительно (в руке или рядом) (сумка)	
# Дополнительно (в руке или рядом) (телефон)	
# Дополнительно (в руке или рядом) (зонт)	
# Дополнительно (в руке или рядом) (дополнительные предметы отсутствуют)	

# Фон (отсутствует)	
# Фон (штриховка)	
# Фон (опорная линия)	
# Фон (заштрихованное облако)	
# Фон (пейзаж)	
# Фон (пейзаж преобладает над изображением человека)	
# Фон (дождь)	
# Фон (солнце)	
# Фон (земля)	
# Фон (другое)	

# Площадь штриховки	

# Нажим	

# Шея отделена от тела линией

In [18]:
# Посчитаем кол-во бинарных признаков

binary_cnt, multilabel_cnt = 0, 0

for feature, values in unique_feature_values.items():
    if len(values) == 1:
        print(f"! ALARM {feature!r} !")
    elif len(values) == 2:
        binary_cnt += 1
    else:
        multilabel_cnt += 1

print(f"Кол-во бинарных признаков: {binary_cnt}")
print(f"Кол-во многоклассовых признаков: {multilabel_cnt}")

! ALARM 'Дополнительные особенности правого глаза (изображены темные очки)' !
! ALARM 'Положение руки (закрывает генитальную область)' !
! ALARM 'Положение руки (прижата к голове)' !
! ALARM 'Нога (отставлена (стандарт))' !
! ALARM 'Нога (короткая)' !
! ALARM 'Нога (длинная)' !
Кол-во бинарных признаков: 264
Кол-во многоклассовых признаков: 40


In [19]:
df.to_excel("2_2_preprocessing of feature labels.xlsx", index=False)